https://www.codewars.com/kata/69876930993d6d1af5107b11/discuss

Ideas.

* Resist attempting to find a mathematical simplification when both $a$ and $b$ in the problem contain $x$ terms - this is not the way I think.
* The problem comprises two parts:
    1. Parsing the mathematical expression.
        - The issue is how to put the mathematical simplificaiton of a terms $7x$ into a computer-friendly $ax^n$ form.
    2. Computing and retrieving the relevant index/coefficient.
* Is there a way to use the mathematics to isolate our term of interest, and then just code that?
    * The issue here mathematically is our $a$ terms and our $b$ terms may both contain our variable $x$.
* Otherwise we would need to create a dictionary of variable powers-coefficients to then lookup.
* Try to see if you can isolate it mathematically, otherwise compute powers.

Mathematical specification.

1. Parsing the mathemtical expression.
    - Th challenge here is to find a clean way to express mathematical abbreviations like $7x$ as $7x^1$, $4$ as $4x$ etc, without getting caught in an imperative mess of `if-else` commands, and using Python built-in functions.
    - Predictability of the expression i.e. always having brackets and a plus\minus suggests that using Regex would help a lot, but I don't currently know Regex.
    - Code this up, and it should motivate you to go and learn Regex, which you will need anyway.
    - HINT: Are there natural partitions of the string that we can use?
    - Solving this for one $ax^n$ generally will automatically yield the other.
    - Some test examples:
        - `(x` means $a = 1$, $n$ = 1.
        - `(7` means $a = 7$, $n$ = 0.
        - `(14x^2` means $a = 14$, $n = 2$.
        - Edge cases are where the mathematical abbreviations make things feel tricky.
    - Write a function that will wrap splitting these strings to encode the mathematical logic.
    - LOGIC.
    - The logic can be dealt with more uniformly using `str.partition("delimiter")`.
    - ...3.5 hours later...I have a cyclomatically complex piece of code, which works under very strict conditions.
    - But only managed to figure out half way through writing it that it was beginning to get very cyclomatically complex - decided to finish it off anyway as it's a good lesson in how NOT to write code, and to better acquaint myself with when code *is* getting cyclomatically complex.
        - The thought process was along the lines of "Working in a modular fashion, we will worry about that edge case later, let's get the main logic out."
        - Bu then each extra variation then yielded more if/else conditionals, which then needed modification of what came before.
        - Spend more time doing an extensive post-code review together with refactoring -- and this is super important.
    
2. Query handling.
    - Construct a `power: coeff` dictionary.

In [1]:
# Submission...of ~3.5 hours' worth of cyclomatically complex spaghetti code :-/

import math 

def extract_parameters(str_expr):

    a = 0
    n = 0
    b = 0
    m = 0
    k = 0
    
    monom_1, operator, monom_2 = str_expr.split()

    # Process 1st monomial.
    coeff_part, x_part, power_part = monom_1.partition("x")

    # Constant monomial case.
    if not x_part:
        a = int(coeff_part.strip("("))

    # 1st monomial contains "x".
    else:

        # Extract monomial coefficient, handling "(-x" and "(x" edge cases.
        if not coeff_part.strip("("):
            a = 1
        elif coeff_part.strip("(") == "-":
            a = -1
        else:
            a = int(coeff_part.strip("("))

        # Extract monomial power, handling "x" edge case as "x^1".
        if not power_part:
            n = 1
        else:
            n = int(power_part.strip("^"))

    # Process 2nd monomial.
    
    # Constant monomial case.
    if "x" not in monom_2:
        
        # Extract the constant and binomial power.
        coeff_part, binom_power_part = monom_2.split(")")
        b = int(coeff_part)
        if not binom_power_part.strip("^"):
            k = 1
        else:
            k = int(binom_power_part.strip("^"))

    # 2nd monomial contains "x".
    else:
        coeff_part, x_part, remainder = monom_2.partition("x")

        # Extract monomial coefficient.
        if not coeff_part:
            b = 1
        elif coeff_part == "-":
            b = -1
        else:
            b = int(coeff_part)

        power_part, binom_power_part = remainder.split(")")

        # Extract monomial power.
        if not power_part.strip("^"):
            m = 1
        else:
            m = int(power_part.strip("^"))

        # Extract binomial power.
        if not binom_power_part.strip("^"):
            k = 1
        else:
            k = int(binom_power_part.strip("^"))

    return a, n, b, m, k, operator

def get_coefficient(binom: str, x_power: int) -> int | float:

    a, n, b, m, k, operator = extract_parameters(binom)

    print(a, n, b, m, k, operator)

    power_coeffs = dict()
    for r in range(k, -1, -1):
        power = (n * r) + (m *(k - r))
        coeff_prod = (a ** r) * ((b ** (k - r)) if operator == "+" else ((-b) ** (k - r)))
        binom_coeff = math.comb(k, r)
        power_coeffs[power] = binom_coeff * coeff_prod

    print(power_coeffs)

    return power_coeffs.get(x_power, 0)

In [22]:
get_coefficient("(5x + 4x)^3", 3)

5 1 4 1 3 +
{3: 64}


64